# Notebook 1: MIMU 特征提取

在每个滑动窗口中提取 MIMU 特征，为每个窗口标记运动类型。

**可行性检查**: t-SNE 散点图验证不同运动类型在特征空间中的可分性。

In [ ]:
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import welch
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path(r"D:\data\PPG_HeartRate\Algorithm\outline-PPGtoHR")
DATA_DIR = PROJECT_ROOT / "20260418test_python"
ARTIFACTS_DIR = PROJECT_ROOT / "docs" / "research" / "artifacts"
sys.path.insert(0, str(PROJECT_ROOT / "python" / "src"))

from ppg_hr.preprocess.data_loader import load_dataset

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

## 1. 运动场景定义和数据文件

In [ ]:
EXERCISES = {
    "jump_rope": {"dir": "tiaosheng", "label": "跳绳", "complexity": "simple"},
    "arm_curl": {"dir": "wanju", "label": "手臂弯举", "complexity": "simple"},
    "push_up": {"dir": "fuwo", "label": "俯卧撑", "complexity": "simple"},
    "jumping_jack": {"dir": "kaihe", "label": "开合跳", "complexity": "simple"},
    "burpee": {"dir": "bobi", "label": "波比跳", "complexity": "complex"},
}

FS = 100
WIN_SEC = 8
STEP_SEC = 1
WIN_SAMPLES = WIN_SEC * FS

In [ ]:
def discover_csv_pairs(scenario_dir: Path) -> list[tuple[Path, Path]]:
    pairs = []
    for csv_file in sorted(scenario_dir.glob("multi_*.csv")):
        if csv_file.stem.endswith("_ref"):
            continue
        ref_file = csv_file.with_name(csv_file.stem + "_ref" + csv_file.suffix)
        if ref_file.is_file():
            pairs.append((csv_file, ref_file))
    return pairs

all_files = {}
for name, info in EXERCISES.items():
    scenario_dir = DATA_DIR / info["dir"]
    pairs = discover_csv_pairs(scenario_dir)
    all_files[name] = pairs
    print(f"{info['label']}: {len(pairs)} 个文件")
    for csv_f, ref_f in pairs:
        print(f"  {csv_f.name}")

## 2. 特征提取函数

In [ ]:
def extract_time_domain_features(signal: np.ndarray) -> dict:
    return {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "min": np.min(signal),
        "max": np.max(signal),
        "range": np.max(signal) - np.min(signal),
        "energy": np.mean(signal ** 2),
        "zcr": np.sum(np.diff(np.sign(signal)) != 0) / (len(signal) - 1),
    }


def extract_freq_domain_features(signal: np.ndarray, fs: int = 100) -> dict:
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 256))
    psd_sum = np.sum(psd)
    if psd_sum == 0:
        return {"dominant_freq": 0.0, "spectral_entropy": 0.0, "spectral_flatness": 0.0}

    non_dc = psd[1:]
    if len(non_dc) > 0 and np.max(non_dc) > 0:
        dominant_freq = freqs[1:][np.argmax(non_dc)]
    else:
        dominant_freq = 0.0

    psd_norm = psd / psd_sum
    psd_norm = psd_norm[psd_norm > 0]
    spectral_entropy = -np.sum(psd_norm * np.log(psd_norm))

    psd_pos = psd[psd > 0]
    if len(psd_pos) > 0:
        spectral_flatness = np.exp(np.mean(np.log(psd_pos))) / np.mean(psd_pos)
    else:
        spectral_flatness = 0.0

    return {
        "dominant_freq": dominant_freq,
        "spectral_entropy": spectral_entropy,
        "spectral_flatness": spectral_flatness,
    }

In [ ]:
MIMU_CHANNELS = ["AccX", "AccY", "AccZ", "GyroX", "GyroY", "GyroZ"]
TIME_FEATURES = ["mean", "std", "min", "max", "range", "energy", "zcr"]
FREQ_FEATURES = ["dominant_freq", "spectral_entropy", "spectral_flatness"]
MAG_FEATURES = ["mean", "std", "energy", "dominant_freq"]


def build_feature_names() -> list[str]:
    names = []
    for ch in MIMU_CHANNELS:
        for feat in TIME_FEATURES:
            names.append(f"{ch}_{feat}")
        for feat in FREQ_FEATURES:
            names.append(f"{ch}_{feat}")
    for mag_name in ["acc_mag", "gyro_mag"]:
        for feat in MAG_FEATURES:
            names.append(f"{mag_name}_{feat}")
    names.extend(["corr_acc_gyro"])
    names.extend(["corr_AccX_AccY", "corr_AccX_AccZ", "corr_AccY_AccZ"])
    names.extend(["corr_GyroX_GyroY", "corr_GyroX_GyroZ", "corr_GyroY_GyroZ"])
    return names

FEATURE_NAMES = build_feature_names()
print(f"特征维度: {len(FEATURE_NAMES)}")

In [ ]:
def extract_window_features(
    accx, accy, accz, gyrox, gyroy, gyroz, fs=100
) -> np.ndarray:
    features = []
    channels = [accx, accy, accz, gyrox, gyroy, gyroz]

    for sig in channels:
        td = extract_time_domain_features(sig)
        fd = extract_freq_domain_features(sig, fs)
        for feat in TIME_FEATURES:
            features.append(td[feat])
        for feat in FREQ_FEATURES:
            features.append(fd[feat])

    acc_mag = np.sqrt(accx**2 + accy**2 + accz**2)
    gyro_mag = np.sqrt(gyrox**2 + gyroy**2 + gyroz**2)

    for mag in [acc_mag, gyro_mag]:
        td = extract_time_domain_features(mag)
        fd = extract_freq_domain_features(mag, fs)
        features.append(td["mean"])
        features.append(td["std"])
        features.append(td["energy"])
        features.append(fd["dominant_freq"])

    def safe_corr(a, b):
        if np.std(a) == 0 or np.std(b) == 0:
            return 0.0
        return np.corrcoef(a, b)[0, 1]

    features.append(safe_corr(acc_mag, gyro_mag))
    features.append(safe_corr(accx, accy))
    features.append(safe_corr(accx, accz))
    features.append(safe_corr(accy, accz))
    features.append(safe_corr(gyrox, gyroy))
    features.append(safe_corr(gyrox, gyroz))
    features.append(safe_corr(gyroy, gyroz))

    return np.array(features, dtype=float)

## 3. 批量提取所有文件的特征

In [ ]:
def extract_features_for_file(csv_path: Path, ref_path: Path) -> pd.DataFrame:
    ds = load_dataset(csv_path, ref_path)
    df = ds.data
    n_samples = len(df)
    win_samples = WIN_SEC * FS
    step_samples = STEP_SEC * FS
    start_idx = int(1.0 * FS)
    buffer_samples = int(10.0 * FS)
    end_idx = n_samples - buffer_samples

    records = []
    for win_start in range(start_idx, end_idx - win_samples + 1, step_samples):
        win_end = win_start + win_samples
        window_time = df["Time_s"].iloc[win_start]
        feat_vec = extract_window_features(
            df["AccX"].iloc[win_start:win_end].to_numpy(),
            df["AccY"].iloc[win_start:win_end].to_numpy(),
            df["AccZ"].iloc[win_start:win_end].to_numpy(),
            df["GyroX"].iloc[win_start:win_end].to_numpy(),
            df["GyroY"].iloc[win_start:win_end].to_numpy(),
            df["GyroZ"].iloc[win_start:win_end].to_numpy(),
            FS,
        )
        records.append({"window_time": window_time, "features": feat_vec})

    return pd.DataFrame(records)


print("开始批量提取特征...")
all_records = []
for name, info in EXERCISES.items():
    pairs = all_files[name]
    for csv_f, ref_f in pairs:
        print(f"  处理: {csv_f.name} ...", end=" ")
        try:
            df_feats = extract_features_for_file(csv_f, ref_f)
            df_feats["exercise"] = name
            df_feats["exercise_label"] = info["label"]
            df_feats["complexity"] = info["complexity"]
            df_feats["file_name"] = csv_f.stem
            all_records.append(df_feats)
            print(f"{len(df_feats)} 个窗口")
        except Exception as e:
            print(f"错误: {e}")

df_all = pd.concat(all_records, ignore_index=True)
print(f"\n总计: {len(df_all)} 个窗口, 来自 {df_all['file_name'].nunique()} 个文件")

In [ ]:
print("各运动类型窗口统计:")
print(df_all.groupby(["complexity", "exercise_label"]).size().to_string())

## 4. 特征矩阵构建

In [ ]:
X = np.stack(df_all["features"].values)
y = df_all["exercise"].values
labels = df_all["exercise_label"].values

nan_mask = ~np.isfinite(X)
if nan_mask.any():
    col_medians = np.nanmedian(X, axis=0)
    for j in range(X.shape[1]):
        mask = nan_mask[:, j]
        X[mask, j] = col_medians[j] if np.isfinite(col_medians[j]) else 0.0
    print(f"已修复 {nan_mask.sum()} 个 NaN/Inf 值")

print(f"特征矩阵: {X.shape}")
print(f"标签分布: {dict(zip(*np.unique(y, return_counts=True)))}")

## 5. 可行性检查: 特征可分性可视化

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA 解释方差比: {pca.explained_variance_ratio_}")
print(f"累计: {pca.explained_variance_ratio_.sum():.2%}")

fig, ax = plt.subplots(figsize=(10, 8))
for name in EXERCISES:
    mask = y == name
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=EXERCISES[name]["label"],
               alpha=0.5, s=20)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA - MIMU 特征空间中的运动类型分布")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 8))
for name in EXERCISES:
    mask = y == name
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=EXERCISES[name]["label"],
               alpha=0.5, s=20)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE - MIMU 特征空间中的运动类型分布")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 简单运动之间的可分性
simple_mask = df_all["complexity"] == "simple"
X_simple = X_scaled[simple_mask]
y_simple = y[simple_mask]

if len(np.unique(y_simple)) > 1:
    tsne_simple = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne_simple = tsne_simple.fit_transform(X_simple)

    fig, ax = plt.subplots(figsize=(10, 8))
    for name in EXERCISES:
        if EXERCISES[name]["complexity"] != "simple":
            continue
        mask = y_simple == name
        ax.scatter(X_tsne_simple[mask, 0], X_tsne_simple[mask, 1],
                   label=EXERCISES[name]["label"], alpha=0.5, s=20)
    ax.set_title("t-SNE - 仅简单运动类型 (训练集)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# 关键特征箱线图
df_feat = pd.DataFrame(X, columns=FEATURE_NAMES)
df_feat["exercise_label"] = labels

key_features = ["acc_mag_std", "gyro_mag_std", "acc_mag_dominant_freq",
                "gyro_mag_dominant_freq", "AccZ_energy", "AccY_energy"]
existing = [f for f in key_features if f in df_feat.columns]

fig, axes = plt.subplots(1, len(existing), figsize=(5 * len(existing), 5))
if len(existing) == 1:
    axes = [axes]
for ax, feat_name in zip(axes, existing):
    sns.boxplot(data=df_feat, x="exercise_label", y=feat_name, ax=ax)
    ax.set_title(feat_name)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. 保存特征数据

In [ ]:
output_path = ARTIFACTS_DIR / "mimu_features_all.pkl"
save_data = {
    "df": df_all[["window_time", "exercise", "exercise_label", "complexity",
                   "file_name", "features"]],
    "X": X,
    "y": y,
    "labels": labels,
    "feature_names": FEATURE_NAMES,
}
with open(output_path, "wb") as f:
    pickle.dump(save_data, f)
print(f"已保存到: {output_path}")
print(f"  特征维度: {X.shape[1]}")
print(f"  总窗口数: {X.shape[0]}")